# Fair Compensation Project

## Split into train and test sets

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

df = pd.read_csv('data/cleaned/modelling_features.csv')
print(f"Starting shape: {df.shape}")

Starting shape: (36450, 96)


In [2]:
# drop features that are not useful for regression
non_features = [
    'job_id',
    'fairness_index',
    'fairness_label',
    'annual_mean',
    'annual_low',
    'annual_high',
]

X = df.drop(columns=non_features)
y = df['fairness_index']


In [3]:
cols_to_scale = [
    'skills_count', 'total_employed', 'bls_annual_mean', 'bls_annual_median', 'bls_p10', 'bls_p90', 'pce_2023_in_millions',
]

# split into test and train 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print()
print("Train fairness index distribution:")
print(y_train.value_counts(normalize=True).round(3) * 100)

X_train: (29160, 90)
X_test:  (7290, 90)

Train fairness index distribution:
fairness_index
0.718735    0.2
1.261635    0.1
0.558153    0.1
0.638876    0.1
0.678805    0.1
           ... 
0.575544    0.0
0.581245    0.0
1.080436    0.0
0.623118    0.0
0.843031    0.0
Name: proportion, Length: 17944, dtype: float64


In [4]:
cols_to_impute = ['total_employed', 'bls_p90']

# fit medians on training data only
train_medians = X_train[cols_to_impute].median()
print("Train medians used for imputation:")
print(train_medians)

# apply to both
X_train[cols_to_impute] = X_train[cols_to_impute].fillna(train_medians)
X_test[cols_to_impute]  = X_test[cols_to_impute].fillna(train_medians)

# check
print("\nNull counts after imputation:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print("All nulls resolved:", X_train.isnull().sum().sum() == 0)

Train medians used for imputation:
total_employed     16390.0
bls_p90           170560.0
dtype: float64

Null counts after imputation:
Series([], dtype: int64)
All nulls resolved: True


## Standardize Numerical Values

In [5]:
scaler = StandardScaler()

# fit ONLY on training data
scaler.fit(X_train[cols_to_scale])

# transform both train and test
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print("Scaled column stats on train (should be ~mean=0, std=1):")
print(X_train[cols_to_scale].describe().round(3))

Scaled column stats on train (should be ~mean=0, std=1):
       skills_count  total_employed  bls_annual_mean  bls_annual_median  \
count     29160.000       29160.000        29160.000          29160.000   
mean         -0.000           0.000            0.000             -0.000   
std           1.000           1.000            1.000              1.000   
min          -1.466          -0.806           -1.949             -2.076   
25%          -0.697          -0.645           -0.703             -0.781   
50%          -0.312          -0.336           -0.151             -0.245   
75%           0.457           0.220            0.401              0.707   
max           6.611           3.976            4.494              4.146   

         bls_p10    bls_p90  pce_2023_in_millions  
count  29160.000  29160.000             29160.000  
mean      -0.000      0.000                -0.000  
std        1.000      1.000                 1.000  
min       -2.320     -2.379                -1.200  
25%    

In [6]:
print("Null counts in feature matrix:")
print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
print()
print("Total rows with any null:", X_train.isnull().any(axis=1).sum())
print("Total rows fully complete:", X_train.notna().all(axis=1).sum())
print()
print("As percentage of training set:")
print((X_train.isnull().sum()[X_train.isnull().sum() > 0] / len(X_train) * 100).round(1))

Null counts in feature matrix:
Series([], dtype: int64)

Total rows with any null: 0
Total rows fully complete: 29160

As percentage of training set:
Series([], dtype: float64)
